# Live Scoreboard Writer v2 (Phase 4b)

Extends the original scoreboard writer with 5 more views:
1. **Time-windowed revenue trend** — revenue per 15-minute window, so you can see trends, not just an ever-climbing total
2. **Recent orders feed** — the last 10 orders, live
3. **Geographic breakdown** — revenue by state
4. **Order fulfillment time** — average minutes from order creation to completion
5. **Status funnel** — reuses the existing status breakdown, just visualized differently on the dashboard side

Same rules as before: run this continuously, in its own kernel, alongside the generator and poller.


In [1]:
!pip install pyarrow -q


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("WooCommerceScoreboardWriterV2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 3.5.5


In [3]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, ArrayType, DoubleType

billing_schema = StructType([
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("city", StringType()),
    StructField("state", StringType()),
    StructField("postcode", StringType()),
    StructField("country", StringType()),
    StructField("email", StringType()),
    StructField("phone", StringType()),
])

line_item_schema = StructType([
    StructField("product_id", LongType()),
    StructField("name", StringType()),
    StructField("quantity", LongType()),
    StructField("subtotal", StringType()),
    StructField("price", DoubleType()),
])

order_schema = StructType([
    StructField("id", LongType()),
    StructField("status", StringType()),
    StructField("currency", StringType()),
    StructField("date_created_gmt", StringType()),
    StructField("date_modified_gmt", StringType()),
    StructField("total", StringType()),
    StructField("customer_id", LongType()),
    StructField("billing", billing_schema),
    StructField("line_items", ArrayType(line_item_schema)),
])

In [4]:
from pathlib import Path

LANDING_PATH = "woo_orders_landing"

SCOREBOARD_DIR = Path("dashboard_scoreboard")
SCOREBOARD_DIR.mkdir(exist_ok=True)
print("Scoreboard folder:", SCOREBOARD_DIR.resolve())

Scoreboard folder: C:\Users\chait\Documents\spark_projects\dashboard_scoreboard


## Base streams
Note `order_level` now also carries `date_created`, `city`, and `state` — needed for the new fulfillment-time and geographic views.

In [5]:
from pyspark.sql.functions import col, to_timestamp, explode

raw_stream = (
    spark.readStream
    .schema(order_schema)
    .option("multiLine", True)
    .json(LANDING_PATH)
)

order_level = raw_stream.select(
    col("id").alias("order_id"),
    col("status"),
    to_timestamp(col("date_created_gmt")).alias("date_created"),
    to_timestamp(col("date_modified_gmt")).alias("date_modified"),
    col("total").cast("double").alias("total"),
    col("billing.city").alias("city"),
    col("billing.state").alias("state"),
)

line_items_flat = raw_stream.select(
    col("id").alias("order_id"),
    explode(col("line_items")).alias("li"),
).select(
    "order_id",
    col("li.name").alias("product_name"),
    col("li.quantity").alias("quantity"),
    col("li.subtotal").cast("double").alias("line_subtotal"),
)

## Aggregations
First two are the originals (unwindowed running totals). The next three are new.

In [6]:
from pyspark.sql.functions import sum as _sum, count as _count, avg as _avg, window, unix_timestamp

# --- existing: revenue and units by product (lifetime running total) ---
product_revenue_agg = line_items_flat.groupBy("product_name").agg(
    _sum("line_subtotal").alias("revenue"),
    _sum("quantity").alias("units_sold"),
)

# --- existing: order count and revenue by status ---
order_status_agg = order_level.groupBy("status").agg(
    _count("*").alias("order_count"),
    _sum("total").alias("revenue"),
)

# --- new: revenue and order count by state (geographic breakdown) ---
state_revenue_agg = order_level.groupBy("state").agg(
    _sum("total").alias("revenue"),
    _count("*").alias("order_count"),
)

# --- new: average fulfillment time for completed orders, as one global number ---
fulfillment_agg = (
    order_level
    .filter(col("status") == "completed")
    .withColumn(
        "fulfillment_minutes",
        (unix_timestamp("date_modified") - unix_timestamp("date_created")) / 60.0,
    )
    .groupBy()
    .agg(
        _avg("fulfillment_minutes").alias("avg_fulfillment_minutes"),
        _count("*").alias("completed_count"),
    )
)

# --- new: revenue per 15-minute time window (the trend line) ---
revenue_trend_windowed = (
    order_level
    .withWatermark("date_modified", "30 minutes")
    .groupBy(window(col("date_modified"), "15 minutes"))
    .agg(
        _sum("total").alias("revenue"),
        _count("*").alias("order_count"),
    )
)

## Scoreboard-writing functions
Two of these (`write_trend`, `write_recent_orders`) keep a small accumulator dictionary/list in memory across batches — that's needed because a single `foreachBatch` call only sees *that trigger's* rows, not the full history, so we build the history up ourselves as batches arrive.

In [7]:
import json
from datetime import datetime

def write_product_scoreboard(batch_df, batch_id):
    pdf = batch_df.toPandas().sort_values("revenue", ascending=False)
    pdf["revenue"] = pdf["revenue"].round(2)
    with open(SCOREBOARD_DIR / "product_revenue.json", "w") as f:
        json.dump(pdf.to_dict(orient="records"), f)
    print(f"[batch {batch_id}] product_revenue.json updated ({len(pdf)} products)")


def write_status_scoreboard(batch_df, batch_id):
    pdf = batch_df.toPandas()
    pdf["revenue"] = pdf["revenue"].round(2)
    total_orders = int(pdf["order_count"].sum()) if len(pdf) else 0
    total_revenue = float(pdf["revenue"].sum()) if len(pdf) else 0.0
    avg_order_value = round(total_revenue / total_orders, 2) if total_orders else 0.0
    scoreboard = {
        "total_orders": total_orders,
        "total_revenue": round(total_revenue, 2),
        "avg_order_value": avg_order_value,
        "by_status": pdf.to_dict(orient="records"),
        "last_updated": datetime.now().isoformat(),
    }
    with open(SCOREBOARD_DIR / "overall_stats.json", "w") as f:
        json.dump(scoreboard, f)
    print(f"[batch {batch_id}] overall_stats.json updated (total_orders={total_orders})")


def write_state_scoreboard(batch_df, batch_id):
    pdf = batch_df.toPandas().sort_values("revenue", ascending=False)
    pdf["revenue"] = pdf["revenue"].round(2)
    with open(SCOREBOARD_DIR / "state_revenue.json", "w") as f:
        json.dump(pdf.to_dict(orient="records"), f)
    print(f"[batch {batch_id}] state_revenue.json updated ({len(pdf)} states)")


def write_fulfillment_scoreboard(batch_df, batch_id):
    pdf = batch_df.toPandas()
    if len(pdf) and pdf.iloc[0]["completed_count"] > 0:
        avg_min = round(float(pdf.iloc[0]["avg_fulfillment_minutes"]), 1)
        completed = int(pdf.iloc[0]["completed_count"])
    else:
        avg_min, completed = None, 0
    result = {
        "avg_fulfillment_minutes": avg_min,
        "completed_count": completed,
        "last_updated": datetime.now().isoformat(),
    }
    with open(SCOREBOARD_DIR / "fulfillment_stats.json", "w") as f:
        json.dump(result, f)
    print(f"[batch {batch_id}] fulfillment_stats.json updated (avg={avg_min} min, n={completed})")


trend_history = {}  # accumulates windows across batches: {window_start_iso: {...}}

def write_trend_scoreboard(batch_df, batch_id):
    pdf = batch_df.toPandas()
    for _, row in pdf.iterrows():
        key = row["window"]["start"].isoformat()
        trend_history[key] = {
            "window_start": row["window"]["start"].isoformat(),
            "window_end": row["window"]["end"].isoformat(),
            "revenue": round(float(row["revenue"]), 2),
            "order_count": int(row["order_count"]),
        }
    records = sorted(trend_history.values(), key=lambda r: r["window_start"])
    with open(SCOREBOARD_DIR / "revenue_trend.json", "w") as f:
        json.dump(records, f)
    print(f"[batch {batch_id}] revenue_trend.json updated ({len(records)} windows)")


recent_orders_history = []  # accumulates individual orders across batches, most recent last

def write_recent_orders(batch_df, batch_id):
    pdf = batch_df.toPandas()
    for _, row in pdf.iterrows():
        recent_orders_history.append({
            "order_id": int(row["order_id"]),
            "status": row["status"],
            "city": row["city"],
            "state": row["state"],
            "total": round(float(row["total"]), 2),
            "date_modified": row["date_modified"].isoformat(),
        })
    recent_orders_history.sort(key=lambda r: r["date_modified"])
    trimmed = recent_orders_history[-10:]
    with open(SCOREBOARD_DIR / "recent_orders.json", "w") as f:
        json.dump(trimmed, f)
    print(f"[batch {batch_id}] recent_orders.json updated ({len(trimmed)} shown, {len(recent_orders_history)} seen total)")

## Start all six continuous jobs
Same as before: `run_seconds = None` means indefinite — leave this running and use Interrupt/Stop when you want to end it. Set a number for a quick test run instead.

In [ ]:
queries = [
    product_revenue_agg.writeStream.outputMode("complete")
        .foreachBatch(write_product_scoreboard).trigger(processingTime="5 seconds").start(),
    order_status_agg.writeStream.outputMode("complete")
        .foreachBatch(write_status_scoreboard).trigger(processingTime="5 seconds").start(),
    state_revenue_agg.writeStream.outputMode("complete")
        .foreachBatch(write_state_scoreboard).trigger(processingTime="5 seconds").start(),
    fulfillment_agg.writeStream.outputMode("complete")
        .foreachBatch(write_fulfillment_scoreboard).trigger(processingTime="5 seconds").start(),
    revenue_trend_windowed.writeStream.outputMode("update")
        .foreachBatch(write_trend_scoreboard).trigger(processingTime="5 seconds").start(),
    order_level.writeStream.outputMode("append")
        .foreachBatch(write_recent_orders).trigger(processingTime="5 seconds").start(),
]

run_seconds = None  # None = run indefinitely; set a number of seconds for a quick test

try:
    if run_seconds is None:
        queries[0].awaitTermination()
    else:
        for q in queries:
            q.awaitTermination(run_seconds)
except KeyboardInterrupt:
    pass
finally:
    for q in queries:
        q.stop()
    print("Stopped.")

[batch 0] overall_stats.json updated (total_orders=968)
[batch 0] product_revenue.json updated (134 products)
[batch 0] state_revenue.json updated (29 states)
[batch 0] recent_orders.json updated (10 shown, 968 seen total)
[batch 0] fulfillment_stats.json updated (avg=11.8 min, n=255)
[batch 0] revenue_trend.json updated (18 windows)
[batch 1] overall_stats.json updated (total_orders=970)
[batch 1] product_revenue.json updated (138 products)
[batch 1] state_revenue.json updated (29 states)
[batch 1] recent_orders.json updated (10 shown, 971 seen total)
[batch 1] fulfillment_stats.json updated (avg=11.7 min, n=256)
[batch 1] revenue_trend.json updated (19 windows)
[batch 2] overall_stats.json updated (total_orders=971)
[batch 2] product_revenue.json updated (141 products)
[batch 2] revenue_trend.json updated (19 windows)
[batch 2] recent_orders.json updated (10 shown, 972 seen total)
[batch 2] fulfillment_stats.json updated (avg=11.7 min, n=257)
[batch 2] state_revenue.json updated (29 